# Hugging Face Transformers Library — Code Companion

This notebook accompanies **Topic: Hugging Face Transformers Library**.

Every notebook in this course sits on top of this library: `AutoTokenizer` loads Llama
3.1's tokenizer, `datasets` loads Alpaca, and `TrainingArguments` configures the
`SFTTrainer`/`DPOTrainer` runs. This notebook walks through the library itself, end to
end, so every piece used elsewhere in the course is demystified in one place.

> **Run this notebook in an environment with internet access** — e.g. Google Colab, or
> your own machine. Every `from_pretrained(...)` call downloads files the first time,
> then caches them locally.

```bash
pip install transformers datasets torch
```

In [ ]:
# Run once if needed:
# !pip install transformers datasets torch

## 1. `pipeline()` — The Easiest Way to Start

`pipeline()` wraps tokenization, model inference, and post-processing into one call.
Three lines get you a fully working, pretrained NLP model.

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("Fine-tuning my own model with Unsloth was surprisingly approachable!")
print(result)
# Expected shape: [{'label': 'POSITIVE', 'score': 0.999...}]

## 2. `AutoTokenizer` & `AutoModel` — The Same Classes Used Throughout This Course

The "Auto" classes inspect the checkpoint name and automatically load the right
tokenizer/model class. This is exactly the pattern behind
`FastLanguageModel.from_pretrained("unsloth/Meta-Llama-3.1-8B")` in the LoRA/QLoRA
notebook — Unsloth's loader wraps these same Auto classes with extra optimizations.

In [ ]:
from transformers import AutoTokenizer, AutoModel

checkpoint = "bert-base-uncased"   # a small model, so this loads quickly for the demo
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

print(f"Loaded tokenizer: {type(tokenizer).__name__}")
print(f"Loaded model:     {type(model).__name__}")

# Same code, different checkpoint -- this is exactly how Topic 9's model-swapping worked
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")
model_gpt2 = AutoModel.from_pretrained("gpt2")
print(f"\nLoaded tokenizer: {type(tokenizer_gpt2).__name__}")
print(f"Loaded model:     {type(model_gpt2).__name__}")

## 3. Tokenizing & Running Inference Manually

This is the full manual path that `pipeline()` automates for you — useful once you need
more control, and exactly what happens conceptually inside every `trainer.train()` call
in the LoRA/QLoRA and DPO notebooks.

In [ ]:
inputs = tokenizer("Transformers are powerful!", return_tensors="pt")
print("Tokenized input IDs:", inputs["input_ids"])

outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state

print("\nOutput shape:", last_hidden_state.shape)
print("-> [batch_size, num_tokens, hidden_dim]")

## 4. The `datasets` Library — Loading Alpaca

Load and preview the exact dataset used in the Data Preparation, LoRA/QLoRA, and
Evaluating LLMs notebooks.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("unsloth/alpaca-cleaned", split="train")
print(dataset)
print("\nFirst training example:")
print(dataset[0])

## 5. `TrainingArguments` — Every Field, Explained

This is the exact class used inside `SFTTrainer` in the LoRA/QLoRA notebook. Rather than
just using it, let's build one field by field and see what each one actually controls.

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir = "./results",              # where checkpoints and logs are written
    per_device_train_batch_size = 2,       # examples processed per GPU, per forward pass
    gradient_accumulation_steps = 4,       # accumulate grads over N steps -> effective batch = 2*4 = 8
    num_train_epochs = 1,                  # how many full passes over the training data
    learning_rate = 2e-4,                  # step size for weight updates
    logging_steps = 10,                    # how often to print/log training loss
    save_strategy = "epoch",               # when to save checkpoints
    optim = "adamw_8bit",                  # memory-efficient optimizer variant
)

print("Effective batch size:",
      args.per_device_train_batch_size * args.gradient_accumulation_steps)
print("Output directory:", args.output_dir)
print("Learning rate:", args.learning_rate)

## 6. Other Popular Ready-Made Pipelines

The same `pipeline()` function covers many tasks — useful for the task-specific
evaluation and LLM-as-judge techniques from the Evaluating LLMs notebook.

In [ ]:
# Named Entity Recognition
ner = pipeline("ner", grouped_entities=True)
print("NER:", ner("Unsloth and Hugging Face are widely used in the open-source AI community."))

In [ ]:
# Extractive Question Answering
qa = pipeline("question-answering")
result = qa(
    question="What technique does LoRA use to reduce trainable parameters?",
    context="LoRA reduces trainable parameters by freezing the pretrained weights and "
            "injecting small low-rank matrices that are trained instead.",
)
print("QA:", result)

## 7. Fine-tuning with `Trainer` (the Non-Unsloth Path)

For comparison with the `SFTTrainer`/`DPOTrainer` used elsewhere in this course, here is
the plain `Trainer` class fine-tuning a small model — no Unsloth, no LoRA, just the base
library. This is what "full fine-tuning" looks like in code, referenced back in the
Full Fine-tuning vs. PEFT notebook.

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

imdb = load_dataset("imdb")
small_train = imdb["train"].shuffle(seed=42).select(range(200)).map(tokenize_fn, batched=True)
small_eval = imdb["test"].shuffle(seed=42).select(range(50)).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results", num_train_epochs=1,
    per_device_train_batch_size=8, logging_steps=10,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=small_train, eval_dataset=small_eval,
)
trainer.train()
print("Full fine-tuning complete (every parameter in `model` was updated).")

## 8. Model Cards & Sharing Your Own Model

Once you have a fine-tuned model — whether from this notebook or the LoRA/QLoRA and DPO
notebooks — sharing it is a single call.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()   # paste an access token from huggingface.co/settings/tokens

# model.push_to_hub("your-username/my-fine-tuned-model")
# tokenizer.push_to_hub("your-username/my-fine-tuned-model")

print("Uncomment the lines above (with your own username/token) to publish your model.")

## Recap & Try It Yourself

You just walked through every library piece used across this course:
- `pipeline()` for instant, working models.
- `AutoTokenizer` / `AutoModel` — the same pattern Unsloth's `FastLanguageModel` wraps.
- Manual tokenization + inference for full control.
- `datasets.load_dataset` loading the exact Alpaca dataset used elsewhere in this course.
- `TrainingArguments`, field by field — the same class configuring `SFTTrainer` and
  `DPOTrainer` in the LoRA/QLoRA and DPO notebooks.
- NER and question-answering pipelines.
- Plain `Trainer` full fine-tuning, for direct comparison against the PEFT approaches
  used throughout the rest of the course.

**Things to try:**
1. Swap `"distilbert-base-uncased"` for `"bert-base-uncased"` in Section 7 and compare
   training time.
2. Try the `"summarization"` or `"translation_en_to_fr"` pipeline tasks.
3. Load a different dataset from the Hub and adapt `tokenize_fn` to its column names.
4. After Section 7, run your fine-tuned model back through `pipeline()` by pointing it at
   the local `./results` checkpoint directory instead of a Hub checkpoint name.